# 1. - Подготовка

| Шаг по ДЗ | Название эксперимента | Датасет для обучения | Датасет для валидации | Описание процесса |
| --- | --- | --- | --- | --- |
| **Шаг 1** | Baseline оценка | *Обучение не проводится* | `val` и `Hard-val` | Замеряем `pass@k` и длину генерации на базовой модели Qwen2.5-0.5B-Instruct без дообучения. Доказываем, что на Hard-val `pass@128=0`. |
| **Шаг 2** | GRPO-only (без gold) | `curriculum+hard_train.pkl` (стандартный, без золотых траекторий) | `val` и `Hard-val` | Обучаем базовую модель только с помощью Curriculum GRPO и reward-функции через верификатор. |
| **Шаг 3.1** | SFT-only (Опционально) | `gold_train_128.pkl` (только собранные gold trajectories) | `val` и `Hard-val` | Обучаем базовую модель классическим Supervised Fine-Tuning на эталонных решениях. В задании сказано, что это полезно для декомпозиции эффектов. |
| **Шаг 3.2** | SFT $\rightarrow$ GRPO | `curriculum+hard_train.pkl` (с reward-функцией) | `val` и `Hard-val` | Берем веса модели, получившиеся на шаге 3.1 (после SFT), и дообучаем их с помощью Curriculum GRPO. |
| **Шаг 4** | SRFT (Выбранный метод) | `curriculum+gold_train.pkl` (смесь on-policy генераций и gold trajectories) | `val` и `Hard-val` | Берем базовую модель (Baseline) и обучаем её гибридным методом, совмещая SFT-сигнал и RL-награды в одном цикле обучения. |

In [ ]:
%cd /content
!rm -rf HW-2_env
!git clone https://github.com/TebelevGt/HW-2_env.git
%cd HW-2_env
# Переключаемся на нужную ветку ПЕРЕД переходом в подпапку rl-shortest-path-agent
!git checkout HW_3_hybrid_rl
%cd rl-shortest-path-agent

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
#%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
import os
import json
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOTrainer, GRPOConfig,SFTTrainer
from envs import (
    get_shortest_path_dataset, 
    correctness_reward_func, 
    reasoning_length_reward_func, 
    format_reward_func,
    ShortestPathDataset
)
from transformers import TrainingArguments 

# 2. GRPO Only model

In [ ]:
def train_grpo_model(
    dataset_path: str,
    output_dir: str = "/kaggle/working/grpo_only",
    num_generations: int = 4,
    model_name: str = "unsloth/Qwen2.5-0.5B-Instruct"
):
    """
    Обучает модель Qwen2.5-0.5B методом GRPO на заданном датасете.
    """

    # --- 1. Конфигурация модели ---
    max_seq_length = 4096 * 2
    lora_rank = 64

    print(f"--- Загрузка модели и токенизатора ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        fast_inference = True,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.8,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = lora_rank * 2,
        lora_dropout = 0.05,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = True,
    )

    tokenizer.padding_side = "left"

    # --- 2. Настройка GRPO ---
    print(f"--- Настройка GRPO Trainer (Dataset: {dataset_path}) ---")
    training_args = GRPOConfig(
        use_vllm = True,
        learning_rate = 5e-6,
        adam_beta1 = 0.9,
        adam_beta2 = 0.99,
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        num_generations = num_generations,
        max_prompt_length = 3000,
        max_completion_length = 1000,
        #max_steps = max_steps,
        #save_steps = max_steps,
        num_train_epochs = 1,
        save_strategy = "epoch",
        max_grad_norm = 0.1,
        report_to = "none",
        output_dir = output_dir,
        shuffle_dataset = False
    )

    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            correctness_reward_func,
            reasoning_length_reward_func,
            format_reward_func
        ],
        args = training_args,
        train_dataset = get_shortest_path_dataset(dataset_path)
    )

    # --- 3. Обучение и сохранение ---
    print(f"--- Начало обучения ---")
    trainer.train()
    
    print(f"--- Сохранение модели в {output_dir} ---")
    trainer.save_model(output_dir)
    
    print(f"--- Сохранение истории обучения ---")
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/training_history.json", "w") as f:
        json.dump(trainer.state.log_history, f)
        
    return model, tokenizer

In [ ]:
ds_curr = ShortestPathDataset.load("data/curriculum/train_curriculum_short.pkl")
ds_hard = ShortestPathDataset.load("data/hard128k/hard_train_128.pkl")
ds_hard.data = ds_hard.data[:200] # Берем только 200 сложных

# Это чтобы и старый курикулом и новый хард трейн
ShortestPathDataset(ds_curr.data + ds_hard.data).save("data/curriculum+hard_train.pkl")

model, tokenizer = train_grpo_model('data/curriculum+hard_train.pkl')

In [ ]:
model.save_pretrained('/kaggle/working/grpo_only_')

# 3. SFT Only

In [ ]:
def train_sft_model(
    dataset_path: str,
    output_dir: str = "/kaggle/working/sft_only",
):
    """
    Обучает модель Qwen2.5-0.5B методом SFT на собранных золотых траекториях.
    """
    max_seq_length = 4096 * 2
    lora_rank = 64

    print(f"--- Загрузка модели и токенизатора ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-0.5B-Instruct",
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        fast_inference = False,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.8,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = lora_rank * 2,
        lora_dropout = 0.05,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = True,
    )

    tokenizer.padding_side = "right" # Для SFT паддинг обычно делают справа

    # --- 1. Подготовка датасета под SFT ---
    print(f"--- Подготовка датасета {dataset_path} ---")
    ds = get_shortest_path_dataset(dataset_path)

    # Превращаем {"prompt": [...], "answer": "..."} в единый текст диалога
    def format_for_sft(example):
        # Собираем полный диалог: system -> user -> assistant (gold answer)
        messages = example["prompt"] + [{"role": "assistant", "content": example["answer"]}]
        # Применяем встроенный шаблон чата Qwen
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        return {"text": text}

    sft_dataset = ds.map(format_for_sft)

    # --- 2. Настройка SFT Trainer ---
    print(f"--- Настройка SFT Trainer ---")
    training_args = TrainingArguments(
        learning_rate = 2e-5, # Для SFT LR обычно чуть выше, чем в RL (2e-5 vs 5e-6)
        adam_beta1 = 0.9,
        adam_beta2 = 0.99,
        weight_decay = 0.01,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        max_steps = 250,
        save_steps = 250,
        #num_train_epochs = 1,
        max_grad_norm = 0.3,
        report_to = "none",
        output_dir = "outputs",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
    )

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = sft_dataset,
        dataset_text_field = "text", # Указываем колонку со склеенным текстом
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = training_args,
    )

    # --- 3. Обучение и сохранение ---
    print(f"--- Начало SFT обучения ---")
    trainer.train()

    print(f"--- Сохранение модели в {output_dir} ---")
    trainer.save_model(output_dir)

    print(f"--- Сохранение истории обучения ---")
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/training_history.json", "w") as f:
        json.dump(trainer.state.log_history, f)
        
    return model, tokenizer


model, tokenizer = train_sft_model('data/hard128k/gold_train_128.pkl', output_dir='/kaggle/working/sft_model')

# 4. SFT + GRPO

In [ ]:
ds_curr = ShortestPathDataset.load("data/curriculum/train_curriculum_short.pkl")
ds_hard = ShortestPathDataset.load("data/hard128k/hard_train_128.pkl")
ds_hard.data = ds_hard.data[:200] # Берем только 200 сложных

# Это чтобы и старый курикулом и новый хард трейн
ShortestPathDataset(ds_curr.data + ds_hard.data).save("data/curriculum+hard_train.pkl")

In [ ]:
model, tokenizer = train_grpo_model(
    dataset_path="data/curriculum+hard_train.pkl",
    output_dir = "/kaggle/working/sft_grpo",
    model_name = "/kaggle/working/sft_model"
)

# 5. Hybrid RL + SFT -> SRFT

In [ ]:
class SRFTTrainer(GRPOTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 💡 ГЛАВНЫЙ ФИКС: Прячем золотые ответы от базового тренера!
        # Извлекаем "answer" и удаляем из словаря inputs. Теперь базовый 
        # GRPOTrainer работает в точности как в вашем Эксперименте 2 и не ломается.
        answers = inputs.pop("answer", None)
        
        # 1. RL часть: вычисляем стандартный GRPO лосс
        grpo_loss = super().compute_loss(model, inputs, return_outputs=False, **kwargs)
        
        sft_loss = 0.0
        # 2. Supervised часть: проверяем, есть ли золотые траектории
        if answers is not None and "prompt" in inputs:
            valid_texts = []
            tokenizer = self.processing_class if hasattr(self, "processing_class") else self.tokenizer
            
            # ВАЖНО: Итерируемся по извлеченным answers
            for p, a in zip(inputs["prompt"], answers):
                # Берем только сложные задачи с золотыми траекториями
                if isinstance(a, str) and "<reasoning>" in a:
                    messages = list(p) + [{"role": "assistant", "content": a}]
                    valid_texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
            
            if valid_texts:
                # Токенизируем золотые ответы
                encoded = tokenizer(
                    valid_texts, padding=True, truncation=True, 
                    max_length=self.args.max_prompt_length + self.args.max_completion_length,
                    return_tensors="pt"
                ).to(model.device)
                
                # Прямой проход (forward) для расчета кросс-энтропии
                sft_outputs = model(**encoded, labels=encoded["input_ids"])
                sft_loss = sft_outputs.loss
        
        # Совмещаем лоссы (коэффициент 0.5 балансирует влияние SFT)
        total_loss = grpo_loss + 0.5 * sft_loss
        
        return (total_loss, None) if return_outputs else total_loss


def train_srft_model(
    dataset_path: str,
    output_dir: str = "/kaggle/working/srft_model",
    num_generations: int = 4,
    model_name: str = "unsloth/Qwen2.5-0.5B-Instruct"
):
    max_seq_length = 2048 # Снижено для защиты от OOM
    lora_rank = 32

    print(f"--- Загрузка модели и токенизатора ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        fast_inference = True, 
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.6, # Оставляем 40% VRAM для расчета SFT градиентов
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = lora_rank * 2,
        lora_dropout = 0.05,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = True,
    )

    tokenizer.padding_side = "left"

    print(f"--- Настройка SRFT Trainer (Dataset: {dataset_path}) ---")
    training_args = GRPOConfig(
        use_vllm = True,
        learning_rate = 5e-6,
        adam_beta1 = 0.9,
        adam_beta2 = 0.99,
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        num_generations = num_generations,
        max_prompt_length = 3000,
        max_completion_length = 1000,
        #max_steps = max_steps,
        #save_steps = max_steps,
        num_train_epochs = 1,
        max_grad_norm = 0.1,
        report_to = "none",
        output_dir = "outputs",
        shuffle_dataset = False,
        remove_unused_columns = False # 💡 ВАЖНО: не удалять колонку "answer", она нужна для SFT!
    )

    # ИСПОЛЬЗУЕМ НАШ КАСТОМНЫЙ ТРЕНЕР
    trainer = SRFTTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            correctness_reward_func,
            reasoning_length_reward_func,
            format_reward_func
        ],
        args = training_args,
        train_dataset = get_shortest_path_dataset(dataset_path)
    )

    print(f"--- Начало SRFT обучения ---")
    trainer.train()

    print(f"--- Сохранение модели в {output_dir} ---")
    trainer.save_model(output_dir)

    return model, tokenizer

In [ ]:
ds_curr = ShortestPathDataset.load("data/curriculum/train_curriculum_short.pkl")
ds_hard = ShortestPathDataset.load("data/hard128k/gold_train_128.pkl")
ds_hard.data = ds_hard.data[:200] # Берем только 200 сложных

# Это чтобы и старый курикулом и новый хард трейн
ShortestPathDataset(ds_curr.data + ds_hard.data).save("data/curriculum+gold_train.pkl")

model, tokenizer = train_srft_model('data/curriculum+gold_train.pkl')